# 🧭 Strategy Notebook: How to Tackle Multi-Table Funnel Projects
### A general framework for projects like "Cool T-Shirts Funnel Analysis"

This notebook isn't about *this specific dataset* — it's the thinking process to reuse
whenever you're handed several tables and asked to analyze a multi-step process
(a signup funnel, an onboarding flow, a checkout process, a hiring pipeline, etc).


## Step 1 — Understand the process before touching data

Before opening a single CSV, write down the **stages** of the process in order, in plain
English. For Cool T-Shirts: *visit → cart → checkout → purchase*. For a SaaS signup
funnel it might be: *ad click → signup → email verified → first login → paid plan*.

Ask:
- What defines "reaching" a stage? (An event? A timestamp? A status flag?)
- Is the process strictly linear, or can users skip stages?
- What's the **unit of analysis** — a user, a session, an order? Whatever it is, that's
  your join key.

## Step 2 — Map each table to a stage

Make a simple table like this one before writing any code. It becomes your join plan.

| Stage | Table | Key column | Event column |
|---|---|---|---|
| Visit | `visits.csv` | `user_id` | `visit_time` |
| Cart | `cart.csv` | `user_id` | `cart_time` |
| Checkout | `checkout.csv` | `user_id` | `checkout_time` |
| Purchase | `purchase.csv` | `user_id` | `purchase_time` |

If any table uses a differently-named key (`id` instead of `user_id`), note it now —
you'll need `left_on`/`right_on` (pandas) or `by = c(...)` (dplyr) later.

## Step 3 — Decide your join type *before* you decide your question

This is the step people get wrong most often. Pick the join based on what you're
measuring, not habit:

- **Measuring drop-off / who's missing?** → left join, stage N into stage N+1, keeping
  everything from the earlier (larger) stage. An inner join here silently deletes the
  very thing you're trying to count.
- **Measuring overlap / who did both things?** → inner join.
- **Reconciling two independent, non-hierarchical lists?** → full/outer join.
- **Stacking equivalent files (e.g. monthly exports)?** → concatenate, don't join at all.

## Step 4 — Build incrementally, one join at a time

Don't chain all your joins in one line on the first try. Build and inspect one merge
at a time:

1. Join stage 1 → stage 2. Check the row count. Does it match stage 1's row count
   (as expected for a left join)?
2. Compute the drop-off percentage for *that* transition. Sanity-check it — does 90%
   drop-off seem plausible, or does it suggest a key mismatch?
3. Only once that's verified, extend the chain to stage 3, then stage 4.

This catches key-mismatch bugs (e.g. wrong case, extra whitespace, wrong column) early,
instead of after you've built a five-table merge and the totals don't add up.

In [ ]:
# Illustrative pattern — verify each join before extending the chain
# step_1_2 = stage1.merge(stage2, how='left', on='key')
# assert len(step_1_2) == len(stage1), "left join should preserve row count"
# print(f"drop-off: {step_1_2['stage2_col'].isnull().mean() * 100:.1f}%")
#
# step_1_3 = step_1_2.merge(stage3, how='left', on='key')
# assert len(step_1_3) == len(stage1)
# ... and so on

## Step 5 — Compute rates relative to a consistent baseline

Decide once: are your percentages "drop-off from the *previous* stage" or "conversion
from the *original* stage"? Both are useful, but mixing them in a report confuses
readers. A funnel table with both, side by side, is usually clearest:

| Stage | Users | % of stage before | % of total |
|---|---|---|---|
| Visit | 2000 | — | 100% |
| Cart | 620 | 31% | 31% |
| Checkout | 410 | 66% | 20.5% |
| Purchase | 290 | 71% | 14.5% |

## Step 6 — Look for the *weakest link*, then ask why

The stage with the biggest percentage drop-off is where a fix has the most leverage —
but the raw percentage doesn't tell you *why* people are leaving. At this point, good
follow-up questions to explore (data permitting):

- Does drop-off vary by **traffic source**, device, or time of day?
- Does drop-off vary by **user segment** (new vs. returning, geography)?
- Is there a **time-based** pattern (e.g. do people abandon carts overnight)?
- Is the drop-off concentrated among a **subset** of users, or spread evenly?

These become natural "Task 13, 14, 15..." extensions once the core funnel is built.

## Step 7 — Sanity-check every number before reporting it

A short checklist to run through before you trust a result:

- [ ] Do stage counts *decrease* monotonically (as expected for a strict funnel)?
- [ ] Does `len(left_join_result) == len(left_table)`?
- [ ] Are there duplicate keys in any table that could be silently fanning out rows?
- [ ] Do percentages sum/relate the way you expect (e.g. drop-off + conversion = 100%)?
- [ ] Does the average/median time-between-stages look physically plausible (not
      negative, not absurdly long)?

## Step 8 — Structure your final write-up

A funnel analysis write-up (or notebook's final markdown cell) should answer, in order:

1. **What's the overall conversion rate**, start to finish?
2. **Where's the biggest leak**, in both absolute and percentage terms?
3. **How long does the successful path take**, typically?
4. **One or two concrete, prioritized recommendations** — tied to the biggest leak,
   not a generic list of "improve UX everywhere."

Keep it to a few sentences. A precise, well-supported paragraph beats a long list of
disconnected observations.

## Quick-reference: question → technique

| If the question is... | Reach for... |
|---|---|
| "How many users reached stage X?" | `len(table_x)` |
| "What % dropped off between X and Y?" | left join X→Y, `.isnull().mean()` |
| "How long between X and Y for users who did both?" | inner join, subtract timestamps |
| "Which segment converts best?" | left join + `.groupby('segment')` |
| "Reassemble a dataset split across files" | `pd.concat` / `bind_rows`, not a join |
| "Two independent lists, keep everyone" | outer/full join |
